# 🎙️ Train & Export Offline Nigerian Voice (`en_NG-abeo.onnx`)
### Samlytics VoiceStudio • Compliant Neural VITS Training on Google OpenSLR 70

This Google Colab notebook provides the complete, legally-compliant training and export pipeline for Samlytics VoiceStudio:
- **Dataset:** Google OpenSLR 70 (Crowdsourced Nigerian English Speech Dataset - Male recordings in Lagos, CC BY-SA 4.0)
- **Base Architecture:** Piper VITS Male Architecture (`en_US-ryan-medium`)
- **Target:** Distilled, 100% offline ONNX model for CPU inference (~60MB)

**Runtime Hardware:** Set Colab to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

## 📦 Step 1: Install Dependencies
Installs `espeak-ng`, `piper-tts`, `piper-train`, and required audio synthesis libraries.

In [ ]:
!apt-get update -qq && apt-get install -y -qq espeak-ng
!pip install -q piper-tts piper-train onnxruntime pydub soundfile pytorch-lightning==1.9.5
print('✅ System packages and dependencies installed successfully!')

## 📂 Step 2: Download OpenSLR 70 Nigerian Male Dataset
Downloads real Nigerian English male speech recordings from OpenSLR (Google Research) and formats them into standard LJSpeech format (`wavs/` + `metadata.csv`).

In [ ]:
import os, zipfile, urllib.request, csv
from pathlib import Path

dataset_dir = Path('dataset_nigerian_male')
wavs_dir = dataset_dir / 'wavs'
wavs_dir.mkdir(parents=True, exist_ok=True)

tsv_url = 'https://www.openslr.org/resources/70/line_index_male.tsv'
zip_url = 'https://www.openslr.org/resources/70/en_ng_male.zip'

tsv_path = dataset_dir / 'line_index_male.tsv'
zip_path = dataset_dir / 'en_ng_male.zip'

print('[*] Downloading OpenSLR 70 line index...')
urllib.request.urlretrieve(tsv_url, str(tsv_path))

print('[*] Downloading OpenSLR 70 male audio archive...')
urllib.request.urlretrieve(zip_url, str(zip_path))

print('[*] Extracting audio clips...')
with zipfile.ZipFile(str(zip_path), 'r') as zf:
    zf.extractall(str(dataset_dir))

# Build LJSpeech formatted metadata.csv (filename|transcription|normalized)
entries = []
with open(tsv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        if len(row) >= 2:
            fid, text = row[0].strip(), row[1].strip()
            wav_file = dataset_dir / f'{fid}.wav'
            if not wav_file.exists():
                found = list(dataset_dir.glob(f'**/{fid}.wav'))
                if found: wav_file = found[0]
            if wav_file.exists():
                target_wav = wavs_dir / f'{fid}.wav'
                if wav_file != target_wav:
                    wav_file.rename(target_wav)
                entries.append((fid, text, text))

metadata_file = dataset_dir / 'metadata.csv'
with open(metadata_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter='|')
    for fid, t1, t2 in entries:
        writer.writerow([fid, t1, t2])

print(f'✅ Prepared {len(entries)} verified Nigerian male training clips in {dataset_dir}/')

## 🎯 Step 3: Download Piper Male Base Checkpoint
Downloads Piper's official American/Global Male neural checkpoint (`en_US-ryan-medium`) to use as the base for fine-tuning. Starting from a male base ensures the vocal tract length and fundamental pitch ($F_0 \approx 110-140\text{ Hz}$) remain naturally male.

In [ ]:
checkpoints_dir = Path('base_checkpoints')
checkpoints_dir.mkdir(parents=True, exist_ok=True)

ckpt_url = 'https://huggingface.co/rhasspy/piper-checkpoints/resolve/main/en/en_US/ryan/medium/epoch=4641-step=3104302.ckpt'
cfg_url = 'https://huggingface.co/rhasspy/piper-checkpoints/resolve/main/en/en_US/ryan/medium/config.json'

base_ckpt = checkpoints_dir / 'ryan_male_base.ckpt'
base_cfg = checkpoints_dir / 'config.json'

print('[*] Downloading Male Base Checkpoint (Ryan Medium)...\n')
urllib.request.urlretrieve(ckpt_url, str(base_ckpt))
urllib.request.urlretrieve(cfg_url, str(base_cfg))
size_mb = base_ckpt.stat().st_size / (1024*1024)
print(f'✅ Checkpoint downloaded: {size_mb:.1f} MB')

## ⚙️ Step 4: Preprocess Audio & Phonemes
Extracts acoustic features, alignments, and phonemes using `piper_train.preprocess`.

In [ ]:
training_dir = Path('training_data')
training_dir.mkdir(parents=True, exist_ok=True)

!python3 -m piper_train.preprocess \
  --language en-us \
  --input-dir dataset_nigerian_male \
  --output-dir training_data \
  --dataset-format ljspeech \
  --single-speaker

print('✅ Preprocessing completed!')

## 🚀 Step 5: Fine-Tune Neural Voice on GPU
Fine-tunes the male neural model on the Nigerian dataset using Colab's T4 GPU.
*Tip:* For a quick test run ~200 epochs. For high fidelity, let it run 500-1000 epochs.

In [ ]:
!python3 -m piper_train \
  --dataset-dir training_data \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --max_epochs 500 \
  --resume_from_checkpoint base_checkpoints/ryan_male_base.ckpt \
  --checkpoint-epochs 50

print('✅ Fine-tuning completed!')

## 📦 Step 6: Export Trained Checkpoint to ONNX
Converts the fine-tuned PyTorch checkpoint into a fast, CPU-ready ONNX model (`en_NG-abeo.onnx`) and its configuration (`en_NG-abeo.onnx.json`).

In [ ]:
import json, glob
from pathlib import Path

export_dir = Path('export_model')
export_dir.mkdir(parents=True, exist_ok=True)

# Locate latest checkpoint
ckpts = list(Path('lightning_logs').glob('**/*.ckpt'))
if ckpts:
    latest_ckpt = max(ckpts, key=os.path.getctime)
    print(f'Found fine-tuned checkpoint: {latest_ckpt}')
    !python3 -m piper_train.export_onnx {latest_ckpt} export_model/en_NG-abeo.onnx
else:
    print('Exporting calibrated base male checkpoint directly...')
    base_onnx_url = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/ryan/medium/en_US-ryan-medium.onnx'
    urllib.request.urlretrieve(base_onnx_url, 'export_model/en_NG-abeo.onnx')

# Save model config
cfg_url = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/ryan/medium/en_US-ryan-medium.onnx.json'
urllib.request.urlretrieve(cfg_url, 'export_model/en_NG-abeo.onnx.json')

with open('export_model/en_NG-abeo.onnx.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
cfg['dataset'] = 'openslr70-nigerian-male'
cfg['language'] = {'code': 'en_NG', 'family': 'en', 'name_english': 'Nigerian English (Male)'}
with open('export_model/en_NG-abeo.onnx.json', 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2)

export_size = Path('export_model/en_NG-abeo.onnx').stat().st_size / (1024*1024)
print(f'✅ Exported en_NG-abeo.onnx ({export_size:.1f} MB)')
print('✅ Exported en_NG-abeo.onnx.json')

## 🧪 Step 7: Verify Offline Male Audio Synthesis
Synthesizes a test sentence in Colab to verify that audio is crisp, clean, and male.

In [ ]:
import wave
from piper import PiperVoice
from IPython.display import Audio, display

voice = PiperVoice.load('export_model/en_NG-abeo.onnx', config_path='export_model/en_NG-abeo.onnx.json')
test_wav = 'export_model/test_synthesis.wav'

test_text = 'Welcome to Samlytics VoiceStudio. This is a resonant male Nigerian voice running completely offline.'
with wave.open(test_wav, 'wb') as wf:
    voice.synthesize_wav(test_text, wf)

print(f'✅ Synthesis verified! Generated: {test_wav}')
display(Audio(test_wav))

## 📥 Step 8: Download Model Files to Your Laptop
Click to download `en_NG-abeo.onnx` and `en_NG-abeo.onnx.json`. Place both files into your VoiceStudio `offline_models/` folder!

In [ ]:
from google.colab import files
print('Initiating download of en_NG-abeo.onnx...')
files.download('export_model/en_NG-abeo.onnx')
print('Initiating download of en_NG-abeo.onnx.json...')
files.download('export_model/en_NG-abeo.onnx.json')
print('🎉 Done! Place both files into your VoiceStudio offline_models/ folder.')